# 🧠 Notebook 14 — Simulation Temporelle, API Clients & Évaluation du Régresseur IA

> **Auteur** : SalesTeam AI Engineering Team  
> **Thèmes couverts** : 
> 1. 📅 **Simulation par Date de Visite (`visit_date`)** : Mécanismes de recalcul dynamique de la récence et de la saisonnalité.
> 2. 👥 **Gestion des Codes Clients dans l'API** : Architecture de `/api/clients` et agrégation des profils.
> 3. ⚖️ **Flux d'Arbitrage IA vs. Historique** : Détection de stabilité via le Coefficient de Variation ($CV$).
> 4. 📐 **Mesure de la Performance** : Distinction fondamentale entre *Divergence Contextuelle* $|Q_{\text{IA}} - Q_{\text{moyenne}}|$ et *Vraie Erreur Prédictive* $|Q_{\text{IA}} - \text{target\_qty}|$.

---

## 👥 1. Comment l'API contient et charge les Codes Clients ?

Dans l'architecture de **SalesTeam AI**, l'API FastAPI expose l'endpoint `GET /api/clients` ([`src/api/routes/clients.py`](../src/api/routes/clients.py)) qui interroge la fonction de service `get_available_clients()` ([`src/services/recommendation.py`](../src/services/recommendation.py)).

### 🔍 Mécanisme d'Extraction :
1. L'API charge la matrice de données `data/processed/training_set.csv`.
2. Elle effectue un groupement par `code_client` (`groupby('code_client')`).
3. Elle calcule pour chaque client :
   - `total_references` : Le nombre total de références produits associées.
   - `commandes_positives` : Le volume total de commandes positives enregistrées (`target_bought.sum()`).
   - `quantite_moyenne` : La quantité moyenne commandée.
4. Les clients sont triés par ordre décroissant d'activité historique et servis au frontend (qui les affiche dans le panneau repliable « Codes clients disponibles »).

In [3]:
import pandas as pd
import numpy as np
import joblib
import json
import os

# 1. Chargement du dataset
DATA_PATH = '../data/processed/training_set.csv'
if not os.path.exists(DATA_PATH):
    DATA_PATH = 'data/processed/training_set.csv'

df_all = pd.read_csv(DATA_PATH)

# 2. Simulation exacte de get_available_clients()
agg_dict = {
    'total_references': ('code_article', 'nunique'),
    'commandes_positives': ('target_bought', 'sum'),
}
if 'avg_qty' in df_all.columns:
    agg_dict['quantite_moyenne'] = ('avg_qty', 'mean')

client_summary = (
    df_all.groupby('code_client')
    .agg(**agg_dict)
    .reset_index()
    .sort_values(by='commandes_positives', ascending=False)
)

print(f"Total clients disponibles dans l'API : {len(client_summary)}")
client_summary.head(10)

Total clients disponibles dans l'API : 569


,code_client,total_references,commandes_positives,quantite_moyenne
52,CLT070730,329,1274,62.086291
75,CLT091206,317,1185,16.637937
5,CLT011712,398,842,8.849508
50,CLT068613,253,718,11.541960
328,CLT109532,300,675,6.371835
401,CLT111330,224,625,36.925381
122,CLT100521,290,574,8.658577
8,CLT023447,252,484,19.661624
209,CLT106195,242,472,5.234316
6,CLT012878,186,438,10.647891


---

## 📅 2. Programmation par Date de Visite (`visit_date`) : Pourquoi et Comment ?

### 🎯 Pourquoi `visit_date` est un signal déterminant ?
Les besoins d'un client B2B ne sont pas statiques ; ils dépendent **du moment exact de la visite** :
1. **Le cycle de réapprovisionnement** : Si le commercial visite le client 10 jours après sa dernière commande, son stock est encore plein (besoin faible). S'il le visite 45 jours après alors que son cycle normal est de 20 jours, le client est en **rupture imminente** (besoin urgent).
2. **La saisonnalité mensuelle** : Certains produits (ex: climatiseurs, smartphones en rentrée scolaire ou fêtes de fin d'année) voient leur demande exploser sur des mois précis.

### ⚙️ Les 4 Variables Recalculées Dynamiquement par le Backend :

Lorsqu'une `visit_date` est transmise dans le payload POST `/api/recommend` :

1. **Régénération de la Date du Dernier Achat (`last_date`)** :
   $$\text{last\_date} = \text{date\_ref\_base} - \text{recency\_days}$$
2. **Nouvelle Récence relative à la Visite (`new_recency_days`)** :
   $$\text{recency\_days} = \max(0, \text{visit\_date} - \text{last\_date})$$
3. **Indice de Retard Relatif (`recency_relative`)** :
   $$\text{recency\_relative} = \frac{\text{recency\_days}}{\text{avg\_delay\_days}}$$
   Ce ratio détermine le **`timing_boost`** :
   - $\ge 1.5$ : **Boost 3.0x** (⚡ *Urgent / Retard critique*)
   - $\ge 1.0$ : **Boost 2.0x** (⚡ *Urgent / Échéance dépassée*)
   - $\ge 0.85$ : **Boost 1.5x** (✅ *Recommandé / Proche du réassort*)
   - $< 0.85$ : **Boost 1.0x** (*Normal / Stock suffisant*)
4. **Coefficient de Saisonnalité Dynamique (`current_month_coef`)** :
   $$\text{current\_month\_coef} = \text{SEASONAL\_COEF}[\text{visit\_date.month}]$$

In [4]:
# Démonstration : Impact du changement de date pour un même client
client_id = 'CLT091206'
df_client = df_all[df_all['code_client'] == client_id].copy().drop_duplicates(subset=['code_article'], keep='last')

default_ref = pd.to_datetime('2026-06-22')
last_dates = default_ref - pd.to_timedelta(df_client['recency_days'], unit='D')

def simuler_visite(target_date_str):
    target_dt = pd.to_datetime(target_date_str)
    new_recency = np.clip((target_dt - last_dates).dt.days, 0, None)
    new_rel = new_recency / df_client['avg_delay_days'].replace(0, 30.0)
    
    # Calcul du boost temporel
    boosts = new_rel.apply(lambda r: 3.0 if r >= 1.5 else (2.0 if r >= 1.0 else (1.5 if r >= 0.85 else 1.0)))
    
    return pd.DataFrame({
        'Article': df_client['code_article'],
        'Délai_Moyen_Jours': df_client['avg_delay_days'].round(1),
        'Récence_Jours': new_recency,
        'Récence_Relative': new_rel.round(2),
        'Timing_Boost': boosts
    }).sort_values('Récence_Relative', ascending=False)

print("📅 SCÉNARIO A : Visite Rapprochée (2026-06-30)")
display(simuler_visite('2026-06-30').head(5))

print("\n📅 SCÉNARIO B : Visite Future Tardive (2026-12-15) -> Rupture & Récence Élevée")
display(simuler_visite('2026-12-15').head(5))

📅 SCÉNARIO A : Visite Rapprochée (2026-06-30)


,Article,Délai_Moyen_Jours,Récence_Jours,Récence_Relative,Timing_Boost
1217695,M2116W1 BLUE,1.0,902,902.00,3.0
1217778,SP01Z07Z1966Y,1.0,396,396.00,3.0
1217648,BG7 WHITE 4/256,6.0,797,132.83,3.0
1217646,BG7 GREEN 4/256,6.0,797,132.83,3.0
1217644,BG7 BLACK 4/256,6.0,797,132.83,3.0



📅 SCÉNARIO B : Visite Future Tardive (2026-12-15) -> Rupture & Récence Élevée


,Article,Délai_Moyen_Jours,Récence_Jours,Récence_Relative,Timing_Boost
1217695,M2116W1 BLUE,1.0,1070,1070.00,3.0
1217778,SP01Z07Z1966Y,1.0,564,564.00,3.0
1217644,BG7 BLACK 4/256,6.0,965,160.83,3.0
1217648,BG7 WHITE 4/256,6.0,965,160.83,3.0
1217646,BG7 GREEN 4/256,6.0,965,160.83,3.0


---

## ⚖️ 3. Flux d'Arbitrage : Prédiction IA vs. Comportement Historique

Lorsqu'un produit est sélectionné par le classifieur, le régresseur IA prédit une quantité $Q_{\text{IA}}$. Le système arbitre alors entre l'IA et la moyenne historique $\bar{Q}$ selon le flux suivant :

```
                       Prédiction IA (Q_IA)
                                │
                                ▼
                     Comparer au comportement
                            historique (Q_moyenne)
                                │
                                ▼
                     Calculer écart relatif
                     |Q_IA - Q_moyenne| / Q_moyenne
                                │
             ┌──────────────────┴──────────────────┐
             │                                     │
        Petit écart                           Grand écart
     (Cohérence forte)                     (Ajustement fort)
             │                                     │
             ▼                                     ▼
       Vérifier le CV                        Vérifier le CV
      (std_qty / avg_qty)                   (std_qty / avg_qty)
             │                                     │
             ├─────────────────┬───────────────────┤
             ▼                 ▼                   ▼
       Client stable     Client stable       Client instable
       (CV <= 1.0)       (CV <= 1.0)         (CV > 1.0)
             │                 │                   │
             ▼                 ▼                   ▼
       ✅ IA Retenue     ✅ IA Retenue       ⚠️ Contrôle Métier / Fallback
     (Haute Confiance) (Clamping Bornes)   (Moyenne Historique Retenue)
```

### 📌 Les 2 Garde-fous Implémentés dans `recommendation.py` :

1. **Le Filtre de Variance ($CV > 1.0$)** :
   - Si un client achète tantôt 5 unités, tantôt 500 unités ($CV > 100\%$), aucun régresseur ne peut prédire avec certitude.
   - **Décision** : Le système sécurise la commande en prenant `ceil(avg_qty)` (`source_quantite = 'historique (variance trop élevée)'`).

2. **Le Clamping Historique (`_clamp_prediction`)** :
   - Même si l'IA s'emballe sur un produit, la quantité suggérée est strictement bornée :
     $$\text{Borne Inférieure} = \max(1, 0.5 \times \text{min\_qty})$$
     $$\text{Borne Supérieure} = 2.0 \times \text{max\_qty}$$
   - Cela garantit qu'aucune quantité aberrante ne soit présentée au commercial.

---

## 📐 4. Évaluation du Modèle : Écart avec la Moyenne vs. Vraie Erreur

> ⚠️ **Règle fondamentale en Data Science & Machine Learning :**
>
> Ne confondez pas **l'écart à la moyenne** et **l'erreur de prédiction** !
>
> * $|Q_{\text{IA}} - Q_{\text{moyenne}}|$ : **Divergence Contextuelle**  
>   Mesure à quel point l'IA s'éloigne des habitudes brutes pour s'adapter à la saisonnalité, la tendance et le délai.
>* $|Q_{\text{IA}} - \text{target\_qty}|$ : **Vraie Erreur Prédictive**  
>   Mesure l'écart entre la recommandation IA et ce que le client a **réellement commandé sur le terrain** (mesuré par la MAE et le RMSE sur l'ensemble de test).

In [ ]:
# Chargement des métadonnées d'entraînement du régresseur
META_PATH = '../src/models/regressor_lsat_metadata.json'
if not os.path.exists(META_PATH):
    META_PATH = 'src/models/regressor_lsat_metadata.json'

with open(META_PATH, 'r', encoding='utf-8') as f:
    meta = json.load(f)

metrics = meta['metrics_rounded']

print("===========================================================")
print("  COMPARAISON DES ERREURS SUR LE JEU DE TEST (RÉEL)")
print("===========================================================")
print(f"1. Erreur Moyenne Absolue (MAE) de la Moyenne Simple : {metrics['mae_baseline']:.2f} unités")
print(f"2. Erreur Moyenne Absolue (MAE) du Régresseur IA      : {metrics['mae']:.2f} unités")
print(f"3. Amélioration apportée par l'IA                    : +{metrics['improvement_mae_pct']:.2f}%")
print("-----------------------------------------------------------")
print(f"4. Erreur Quadratique (RMSE) de la Moyenne Simple     : {metrics['rmse_baseline']:.2f}")
print(f"5. Erreur Quadratique (RMSE) du Régresseur IA         : {metrics['rmse']:.2f}")
print(f"6. Amélioration RMSE                                  : +{metrics['improvement_rmse_pct']:.2f}%")
print("===========================================================")

### 💡 Interprétation des Résultats Chiffrés :

1. **Pourquoi l'IA bat la moyenne ?**
   - Si un client achète en moyenne 10 unités par commande sur l'année, mais qu'en décembre il en achète 15 et en février 6 :
     - Prédire la moyenne (10) fait une erreur de 5 en décembre et 4 en février.
     - Le régresseur IA, grâce à `current_month_coef` et `last_qty`, prédit 14 en décembre et 7 en février $\rightarrow$ son erreur moyenne tombe à **6.53 unités contre 7.14 pour la moyenne**.

2. **Que faire quand la différence est minime (ex: 1 à 2 unités) ?**
   - Si $Q_{\text{IA}} = 12$ et $Q_{\text{moyenne}} = 10$, l'écart est léger.
   - **C'est le cas idéal** : l'IA respecte les ordres de grandeur habituels du client tout en capturant le petit surplus justifié par le contexte actuel.

---

## 🏁 5. Synthèse Récapitulative

| Question Clé | Réponse & Logique Système |
|---|---|
| **Comment l'API fournit les clients ?** | Via `GET /api/clients`, groupement agrégé sur `training_set.csv` trié par volume d'achats. |
| **Quel est le rôle de la `visit_date` ?** | Recalcul temps réel de `recency_days`, du `timing_boost` d'urgence et du coefficient saisonnier du mois. |
| **Quand retient-on l'IA vs la moyenne ?** | Si $CV \le 1.0$ $\rightarrow$ IA retenue (et clampée). Si $CV > 1.0$ $\rightarrow$ moyenne historique sécurisée. |
| **Comment mesurer la vraie qualité de l'IA ?** | Par la MAE/RMSE par rapport aux achats réels ($|Q_{\text{IA}} - \text{target\_qty}|$), et non par rapport à la moyenne. |